# 08 - Day 4: Explainability / Unseen-Attack Analysis

**Day 4 is analysis-only. No model is retrained and no threshold is tuned using Friday data.**

This notebook explains Day 3's zero-day results: why the Random Forest
generalizes well to DDoS, moderately to PortScan, poorly to Bot, and why
the hybrid detector performs extremely poorly across all three. It does
not fit, retrain, or tune anything -- it loads the already-frozen Day 1
RF, Day 2 IsolationForest, and Day 2 hybrid configuration exactly as
Day 3 evaluated them, and analyzes feature-distribution shift and each
detector's score distribution.

**Requires** (already produced by Day 1/2/3, not regenerated here):
- `data/processed/day1/train.parquet`, `test.parquet`, `split_metadata.json`
- `models/day1/random_forest_baseline.joblib`
- `models/day2/isolation_forest.joblib`, `models/day2/day2_metadata.json`
- `results/day2/validation_threshold_selection.json`
- `results/day3/zero_day_comparison.csv`, `results/day3/zero_day_summary.json`


## 1. Research objective

Why does the Random Forest generalize well to some unseen attacks (DDoS), moderately to PortScan, but poorly to Bot, while the Hybrid detector performs extremely poorly on all three? This notebook investigates via feature-distribution shift and score-distribution analysis -- it does not retrain, tune, or re-select anything.

## 2. Experimental integrity

Setup, artifact loading, and integrity checks (pre-fitted models, feature alignment, no unseen-class leakage).

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.utils.validation import check_is_fitted

from src.day2.anomaly import AnomalyModel, anomaly_scores
from src.day2.hybrid import HybridConfig, combine_scores, hybrid_predict
from src.day2.thresholding import RF_DAY1_FROZEN_THRESHOLD
from src.day3.zero_day import assert_feature_alignment, assert_no_unseen_leakage, class_inventory
from src.day4.analysis import (
    build_class_feature_shift_table,
    detection_vs_shift_table,
    distribution_shift_tests,
    group_statistics,
    hybrid_threshold_crossing_table,
    rf_feature_importances,
    top_n_features,
    top_shifted_features_per_class,
)

DATA_DIR = PROJECT_ROOT / "data" / "processed" / "day1"
DAY1_MODEL_DIR = PROJECT_ROOT / "models" / "day1"
DAY2_MODEL_DIR = PROJECT_ROOT / "models" / "day2"
DAY2_RESULTS_DIR = PROJECT_ROOT / "results" / "day2"
DAY3_RESULTS_DIR = PROJECT_ROOT / "results" / "day3"
RESULTS_DIR = PROJECT_ROOT / "results" / "day4"
FIGURES_DIR = RESULTS_DIR / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TOP_N_FEATURES = 20
TOP_N_SHIFTED_PER_CLASS = 10
SAMPLE_CAP = 20_000
RANDOM_STATE = 42
FALLBACK_IF_THRESHOLD = 0.15
FALLBACK_HYBRID_THRESHOLD = 0.50

print("Day 4 is analysis-only. No model is retrained and no threshold is tuned using Friday data.")


In [ ]:
required = {
    "train": DATA_DIR / "train.parquet",
    "test": DATA_DIR / "test.parquet",
    "metadata": DATA_DIR / "split_metadata.json",
    "rf_model": DAY1_MODEL_DIR / "random_forest_baseline.joblib",
    "if_model": DAY2_MODEL_DIR / "isolation_forest.joblib",
}
missing = {k: str(v) for k, v in required.items() if not v.exists()}
if missing:
    raise FileNotFoundError(f"Day 4 requires existing Day 1/Day 2 artifacts. Missing: {missing}")

metadata = json.loads(required["metadata"].read_text())
feature_names = metadata["feature_names"]

train_df = pd.read_parquet(required["train"])
test_df = pd.read_parquet(required["test"])
rf_model = joblib.load(required["rf_model"])
if_raw_model = joblib.load(required["if_model"])

check_is_fitted(rf_model)
check_is_fitted(if_raw_model)
assert_feature_alignment(feature_names, rf_model, context="Random Forest baseline")
assert_feature_alignment(feature_names, if_raw_model, context="IsolationForest")
print("Integrity checks passed: RF and IsolationForest are pre-fitted, loaded, feature-aligned models.")


## 3. Unseen-class verification

Reuses `src.day3.zero_day.class_inventory` (not reimplemented) to confirm Bot, DDoS, and PortScan are absent from the full Monday-Thursday training period.

In [ ]:
inventory = class_inventory(train_df, test_df)
assert_no_unseen_leakage(train_df, inventory.unseen_attack_classes)

expected_unseen = {"Bot", "DDoS", "PortScan"}
actual_unseen = set(inventory.unseen_attack_classes)
unseen_classes = sorted(expected_unseen.intersection(actual_unseen)) or inventory.unseen_attack_classes

print("Unseen classes verified absent from training:", unseen_classes)
print(json.dumps(inventory.summary(), indent=2))


## 4. Day 3 performance recap

Loads Day 3's existing results -- not recomputed here.

In [ ]:
day3_comparison_path = DAY3_RESULTS_DIR / "zero_day_comparison.csv"
day3_summary_path = DAY3_RESULTS_DIR / "zero_day_summary.json"
if not day3_comparison_path.exists() or not day3_summary_path.exists():
    raise FileNotFoundError(
        f"Day 4 requires existing Day 3 results in {DAY3_RESULTS_DIR}. Run scripts/run_day3.py first."
    )

day3_comparison = pd.read_csv(day3_comparison_path)
day3_summary = json.loads(day3_summary_path.read_text())
day3_comparison


## 5. RF feature importance

Reads the already-fitted Random Forest's `feature_importances_` -- does not refit.

In [ ]:
importances = rf_feature_importances(rf_model, feature_names)
top_features = top_n_features(importances, TOP_N_FEATURES)
importances.head(TOP_N_FEATURES)


## 6. Class-specific feature distributions

Descriptive statistics (mean, std, median, 25%, 75%) for the top features across Monday-Thursday training, Friday benign, and each Friday unseen class.

In [ ]:
label_col = "label_multiclass"
benign_df = test_df[test_df[label_col] == inventory.benign_label]
group_dfs = {"benign": benign_df}
for cls in unseen_classes:
    group_dfs[cls.lower()] = test_df[test_df[label_col] == cls]

train_stats = group_statistics(train_df, top_features)
group_stats = {name: group_statistics(gdf, top_features) for name, gdf in group_dfs.items()}

train_stats.head(TOP_N_FEATURES)


## 7. Feature-shift analysis

Class-specific mean shift relative to training for every top feature -> `day4_class_feature_shift.csv`, plus top shifted features per class -> `day4_top_shifted_features.csv`.

In [ ]:
shift_table = build_class_feature_shift_table(importances, train_stats, group_stats, top_features)
shift_table.to_csv(RESULTS_DIR / "day4_class_feature_shift.csv", index=False)

group_names_lower = [c.lower() for c in unseen_classes]
top_shifted = top_shifted_features_per_class(shift_table, group_names_lower, n=TOP_N_SHIFTED_PER_CLASS)
top_shifted.to_csv(RESULTS_DIR / "day4_top_shifted_features.csv", index=False)

shift_table.head(TOP_N_FEATURES)


In [ ]:
detection_rates = {}
test_medians_for_impute = train_df[feature_names].median(numeric_only=True)
X_test = test_df[feature_names].fillna(test_medians_for_impute)
assert not X_test.isna().any().any()
assert not np.isinf(X_test.to_numpy()).any()
train_medians = test_medians_for_impute


## 8. Statistical shift analysis

Mann-Whitney U test (training vs. each Friday group) for each top feature, with a documented sample cap and both a rank-biserial effect size and Cohen's d -> `day4_distribution_tests.csv`. This tests whether distributions differ; it does not by itself establish causality.

In [ ]:
dist_tests = distribution_shift_tests(
    train_df, group_dfs, top_features, sample_cap=SAMPLE_CAP, random_state=RANDOM_STATE
)
dist_tests.to_csv(RESULTS_DIR / "day4_distribution_tests.csv", index=False)
dist_tests.sort_values("p_value").head(15)


## 9-11. Score distributions (RF / IsolationForest / Hybrid)

Score each detector once on Friday using its already-frozen threshold, then plot benign vs. each unseen class.

In [ ]:
if_config = {}
day2_metadata_path = DAY2_MODEL_DIR / "day2_metadata.json"
if day2_metadata_path.exists():
    if_config = json.loads(day2_metadata_path.read_text()).get("isolation_forest_config", {})

anomaly_model = AnomalyModel(
    model=if_raw_model, feature_names=list(feature_names), train_medians=train_medians,
    contamination=if_config.get("contamination", if_raw_model.get_params().get("contamination")),
    n_estimators=if_config.get("n_estimators", if_raw_model.get_params().get("n_estimators")),
    random_state=if_config.get("random_state", if_raw_model.get_params().get("random_state")),
    n_training_samples=if_config.get("n_training_samples", -1),
)

day2_threshold_path = DAY2_RESULTS_DIR / "validation_threshold_selection.json"
rf_threshold = RF_DAY1_FROZEN_THRESHOLD
if day2_threshold_path.exists():
    day2_thresholds = json.loads(day2_threshold_path.read_text())
    if_threshold = day2_thresholds["isolation_forest"]["selected_threshold"]
    hybrid_threshold = day2_thresholds["hybrid"]["selected_threshold"]
    threshold_source = str(day2_threshold_path)
else:
    print(f"WARNING: {day2_threshold_path} not found -- using documented fallback thresholds.")
    if_threshold = FALLBACK_IF_THRESHOLD
    hybrid_threshold = FALLBACK_HYBRID_THRESHOLD
    threshold_source = "fallback_default"

hybrid_config = HybridConfig(threshold=hybrid_threshold)

test_proba = rf_model.predict_proba(X_test)[:, 1]
test_anomaly = anomaly_scores(anomaly_model, test_df)
test_hybrid = combine_scores(test_proba, test_anomaly, hybrid_config)

test_pred_rf = (test_proba >= rf_threshold).astype(int)
test_pred_if = (test_anomaly >= if_threshold).astype(int)
test_pred_hybrid = hybrid_predict(test_hybrid, hybrid_threshold)

print(f"RF threshold={rf_threshold}, IF threshold={if_threshold}, Hybrid threshold={hybrid_threshold} (source: {threshold_source})")


In [ ]:
def score_dist_plot(scores, threshold, title, xlabel, out_path):
    fig, ax = plt.subplots(figsize=(9, 5))
    for cls in ["Benign"] + unseen_classes:
        mask = (test_df[label_col] == cls).to_numpy()
        if mask.sum() == 0:
            continue
        ax.hist(scores[mask], bins=40, alpha=0.55, label=cls, density=True)
    ax.axvline(threshold, color="black", linestyle="--", label=f"Frozen threshold = {threshold}")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()

score_dist_plot(
    test_proba, rf_threshold,
    "Random Forest score distribution -- Benign vs. unseen classes (Friday)",
    "RF attack probability", FIGURES_DIR / "rf_score_by_unseen_class.png",
)


In [ ]:
score_dist_plot(
    test_anomaly, if_threshold,
    "IsolationForest score distribution -- Benign vs. unseen classes (Friday)",
    "Normalized anomaly score", FIGURES_DIR / "if_score_by_unseen_class.png",
)


In [ ]:
score_dist_plot(
    test_hybrid, hybrid_threshold,
    "Hybrid score distribution -- Benign vs. unseen classes (Friday)",
    "Hybrid score", FIGURES_DIR / "hybrid_score_by_unseen_class.png",
)


## 12. Hybrid failure analysis

Exactly how many samples from each unseen class cross the frozen hybrid threshold -> `day4_hybrid_threshold_analysis.csv`. This is the most direct evidence for why the hybrid detects almost none of the unseen attacks.

In [ ]:
hybrid_scores_by_class = {cls: test_hybrid[(test_df[label_col] == cls).to_numpy()] for cls in unseen_classes}
hybrid_threshold_analysis = hybrid_threshold_crossing_table(hybrid_scores_by_class, hybrid_threshold)
hybrid_threshold_analysis.to_csv(RESULTS_DIR / "day4_hybrid_threshold_analysis.csv", index=False)
hybrid_threshold_analysis


## 13. Detector comparison

DDoS / PortScan / Bot detection rate, RF vs. IsolationForest vs. Hybrid, at their respective frozen thresholds.

In [ ]:
detection_rates = {}
for cls in unseen_classes:
    cls_mask = (test_df[label_col] == cls).to_numpy()
    detection_rates[cls.lower()] = {
        "random_forest": float(test_pred_rf[cls_mask].mean()) if cls_mask.any() else float("nan"),
        "isolation_forest": float(test_pred_if[cls_mask].mean()) if cls_mask.any() else float("nan"),
        "hybrid": float(test_pred_hybrid[cls_mask].mean()) if cls_mask.any() else float("nan"),
    }

fig, ax = plt.subplots(figsize=(8, 5))
width = 0.25
x = np.arange(len(unseen_classes))
for i, detector in enumerate(["random_forest", "isolation_forest", "hybrid"]):
    vals = [detection_rates[c.lower()][detector] for c in unseen_classes]
    ax.bar(x + (i - 1) * width, vals, width, label=detector)
ax.set_xticks(x)
ax.set_xticklabels(unseen_classes)
ax.set_ylabel("Detection rate (recall)")
ax.set_ylim(0, 1.05)
ax.set_title("Unseen-attack detection rate by detector (frozen thresholds)")
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / "unseen_attack_detection_comparison.png", dpi=150)
plt.show()

detection_vs_shift = detection_vs_shift_table(shift_table, group_names_lower, detection_rates)
detection_vs_shift.to_csv(RESULTS_DIR / "day4_detection_vs_shift.csv", index=False)
detection_vs_shift


## 14. Feature importance vs. distribution shift

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for cls_lower in group_names_lower:
    col = f"{cls_lower}_shift_pct"
    if col not in shift_table.columns:
        continue
    ax.scatter(shift_table[col].abs(), shift_table["RF_importance"], alpha=0.7, label=cls_lower)
ax.set_xlabel("Absolute feature mean shift vs. training (%)")
ax.set_ylabel("RF feature importance")
ax.set_title("RF feature importance vs. Friday feature-distribution shift (top features)")
ax.legend(title="Unseen class")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "feature_importance_vs_shift.png", dpi=150)
plt.show()


## 15. Feature-shift heatmap

In [ ]:
heatmap_cols = [f"{c}_shift_pct" for c in ["benign"] + group_names_lower if f"{c}_shift_pct" in shift_table.columns]
heatmap_data = shift_table.set_index("feature")[heatmap_cols]
fig, ax = plt.subplots(figsize=(8, max(6, 0.3 * len(heatmap_data))))
im = ax.imshow(heatmap_data.to_numpy(), aspect="auto", cmap="coolwarm", vmin=-200, vmax=200)
ax.set_xticks(range(len(heatmap_cols)))
ax.set_xticklabels([c.replace("_shift_pct", "") for c in heatmap_cols], rotation=45, ha="right")
ax.set_yticks(range(len(heatmap_data)))
ax.set_yticklabels(heatmap_data.index)
ax.set_title("Feature mean-shift vs. training (%) -- top RF-important features")
fig.colorbar(im, ax=ax, label="Mean shift (%)")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "class_feature_shift_heatmap.png", dpi=150)
plt.show()


## 16. Research interpretation

Scientifically conservative -- grounded only in the numbers computed above. Does not claim feature drift *caused* the model failure; reports it as an observed association.

In [ ]:
summary_rows = []
for cls in unseen_classes:
    cls_lower = cls.lower()
    n_samples = int((test_df[label_col] == cls).sum())
    summary_rows.append({
        "attack_class": cls, "samples": n_samples,
        "rf_detection_rate": detection_rates[cls_lower]["random_forest"],
        "if_detection_rate": detection_rates[cls_lower]["isolation_forest"],
        "hybrid_detection_rate": detection_rates[cls_lower]["hybrid"],
        "mean_abs_feature_shift_pct": float(shift_table[f"{cls_lower}_shift_pct"].abs().mean()),
    })
unseen_summary = pd.DataFrame(summary_rows)
unseen_summary.to_csv(RESULTS_DIR / "day4_unseen_attack_summary.csv", index=False)

ordered_by_rf = unseen_summary.sort_values("rf_detection_rate", ascending=False)["attack_class"].tolist()

interpretation = {
    "summary_ordering_by_rf_detection": ordered_by_rf,
    "findings": [
        f"Among the unseen classes, Random Forest detection rate is highest for {ordered_by_rf[0]} and lowest for {ordered_by_rf[-1]}, consistent with the Day 3 zero-day results.",
        "Binary attack/benign detection by the Random Forest does not mean it can identify the unseen class by name -- it was never trained with any label for these classes, only a benign-vs-attack signal.",
        "The hybrid detector's detection rate is lower than both the Random Forest's and IsolationForest's individually on every unseen class analyzed here -- a negative result, reported as-is rather than tuned away.",
        "The hybrid score distribution and its frozen threshold (see day4_hybrid_threshold_analysis.csv and hybrid_score_by_unseen_class.png) show most unseen-attack rows falling below the frozen 0.5 threshold, which is consistent with the RF-weighted (0.7) hybrid formula inheriting the Random Forest's reduced attack-probability confidence on this temporally-shifted traffic, pulling the combined score down even where the anomaly component alone was comparatively higher.",
        "Feature-distribution shift between Monday-Thursday training and Friday is present across several top-importance features (see day4_class_feature_shift.csv and day4_distribution_tests.csv) and is associated with reduced Random Forest confidence on temporally-later traffic. This is reported as an observed association, not as an established causal mechanism.",
        "Temporal distribution shift is a plausible contributing factor to reduced detector generalization and is a reasonable direction for further investigation, but this Day 4 analysis alone does not prove that shift is the sole cause of the differences observed across DDoS, PortScan, and Bot.",
    ],
    "caveats": [
        "This is a descriptive/associative analysis. Mann-Whitney U tests and effect sizes establish that distributions differ; they do not, by themselves, establish that a given feature's shift caused any specific detector's error on any specific row.",
        "No model was retrained. No threshold was tuned using Friday data or unseen-class labels. All thresholds and model artifacts are reused exactly as frozen in Day 1/Day 2/Day 3.",
    ],
}
(RESULTS_DIR / "day4_research_interpretation.json").write_text(json.dumps(interpretation, indent=2, default=str))

print("=== Day 4 Research Interpretation ===")
for line in interpretation["findings"]:
    print("-", line)


## 17. Paper-ready conclusions

- The Random Forest's binary attack/benign signal generalizes unevenly across unseen attack classes on Friday, with the strongest signal for DDoS, an intermediate signal for PortScan, and the weakest for Bot -- but it cannot name any of these classes, since it was trained without labels for them.
- IsolationForest, fit only on benign Monday-Thursday traffic, provides an independent anomaly signal that does not require having seen an attack class, but its raw recall on these unseen classes is lower than the Random Forest's in this dataset.
- The hybrid detector (0.7 x RF + 0.3 x anomaly, frozen threshold 0.5) underperforms both individual detectors on every unseen class analyzed. This is a genuine negative result: the RF-dominated weighting means the hybrid score inherits the Random Forest's reduced confidence on temporally-shifted Friday traffic, and the frozen threshold was calibrated on Thursday validation data where that degradation had not yet been observed.
- Feature-distribution shift across several top-importance features between training and Friday is observed and is associated with, though not proven to cause, the detectors' reduced confidence on unseen-class traffic -- a plausible, evidence-supported explanation rather than an established causal claim.
- These findings support treating threshold freezing, feature-shift monitoring, and score-combination weighting as open problems for future work, rather than treating Day 2's hybrid formula as production-ready as-is.


## Metadata

In [ ]:
day4_metadata = {
    "datasets_used": {"train": str(required["train"]), "test_friday": str(required["test"])},
    "models_used": {"random_forest": str(required["rf_model"]), "isolation_forest": str(required["if_model"])},
    "unseen_classes": unseen_classes,
    "frozen_thresholds": {
        "random_forest": rf_threshold, "isolation_forest": if_threshold,
        "hybrid": hybrid_threshold, "source": threshold_source,
    },
    "hybrid_config": hybrid_config.summary(),
    "features_analyzed": top_features,
    "n_features_analyzed": len(top_features),
    "statistical_methodology": {
        "test": "Mann-Whitney U (two-sided)",
        "effect_sizes": ["rank_biserial_correlation", "cohens_d"],
        "sample_cap": SAMPLE_CAP, "random_state": RANDOM_STATE,
    },
    "integrity_checks_passed": [
        "rf_and_if_pre_fitted_before_use", "feature_alignment_rf", "feature_alignment_isolation_forest",
        "no_unseen_class_in_training_data", "no_nan_or_inf_after_imputation",
    ],
    "no_models_retrained": True,
    "no_friday_labels_used_for_tuning": True,
    "generated_files": [
        "day4_class_feature_shift.csv", "day4_top_shifted_features.csv", "day4_detection_vs_shift.csv",
        "day4_hybrid_threshold_analysis.csv", "day4_distribution_tests.csv", "day4_unseen_attack_summary.csv",
        "day4_research_interpretation.json", "day4_metadata.json",
        "figures/rf_score_by_unseen_class.png", "figures/if_score_by_unseen_class.png",
        "figures/hybrid_score_by_unseen_class.png", "figures/feature_importance_vs_shift.png",
        "figures/class_feature_shift_heatmap.png", "figures/unseen_attack_detection_comparison.png",
    ],
}
(RESULTS_DIR / "day4_metadata.json").write_text(json.dumps(day4_metadata, indent=2, default=str))
print("Day 4 outputs written to:", RESULTS_DIR)
